# Amazon Bedrock AgentCore Runtime에 MCP Server 호스팅 - OAuth Inbound Authentication

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime에 MCP(Model Context Protocol) server를 호스팅하는 방법을 알아봅니다. Amazon Bedrock AgentCore Python SDK를 사용하여 MCP tool을 Amazon Bedrock AgentCore와 호환되는 MCP server로 래핑합니다.

Amazon Bedrock AgentCore Python SDK가 MCP server 구현 세부 사항을 처리하므로 tool의 핵심 기능에 집중할 수 있습니다. 이 SDK는 직접 통신할 수 있도록 코드를 AgentCore의 표준화된 MCP protocol contract로 변환합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Tool 호스팅                                               |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 MCP server 호스팅                    |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 쉬움                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 MCP                |

### 튜토리얼 아키텍처

이 튜토리얼에서는 MCP server를 AgentCore Runtime에 배포하는 방법을 설명합니다.

시연을 위해 `add_numbers`, `multiply_numbers`, `greet_user`의 세 가지 tool이 포함된 간단한 MCP server를 사용합니다.

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### 튜토리얼 주요 기능

* custom tool로 MCP server 생성
* 로컬에서 MCP server 테스트
* Amazon Bedrock AgentCore Runtime에 MCP server 호스팅
* 인증을 사용하여 배포된 MCP server 호출


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials 구성 완료
* Amazon Bedrock AgentCore SDK
* MCP(Model Context Protocol) library
* 실행 중인 Docker

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

ssm_client = boto_session.client("ssm", region_name=region)
secrets_client = boto_session.client("secretsmanager", region_name=region)
agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)

tool_name = "mcp_server_agentcore"

## MCP(Model Context Protocol) 이해

MCP는 AI 모델이 외부 데이터와 tool에 안전하게 액세스하도록 지원하는 protocol입니다. 주요 개념은 다음과 같습니다.

* **Tools**: AI가 작업을 수행하기 위해 호출할 수 있는 함수
* **Streamable HTTP**: AgentCore Runtime에서 사용하는 transport protocol
* **Session Isolation**: 각 client가 `Mcp-Session-Id` header를 통해 격리된 session을 사용
* **Stateless Operation**: 확장성을 위해 server가 stateless operation을 지원해야 함

AgentCore Runtime은 MCP server가 기본 path인 `0.0.0.0:8000/mcp`에 호스팅되기를 기대합니다.

### 프로젝트 구조

프로젝트를 다음과 같은 구조로 설정합니다.

```
mcp_server_project/
├── mcp_server.py              # 주요 MCP server 코드
├── my_mcp_client.py          # 로컬 test client
├── my_mcp_client_remote.py   # 원격 test client
├── requirements.txt          # 의존성
└── __init__.py              # Python package 표시 파일
```

## MCP Server 생성

간단한 tool 세 개가 포함된 MCP server를 생성합니다. 이 server는 AgentCore Runtime 호환성에 필요한 `stateless_http=True` 설정으로 FastMCP를 사용합니다.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### 코드 동작 설명

* **FastMCP**: tool을 호스팅할 수 있는 MCP server 생성
* **@mcp.tool()**: Python 함수를 MCP tool로 변환하는 decorator
* **stateless_http=True**: AgentCore Runtime 호환성에 필요
* **Tools**: 서로 다른 유형의 작업을 보여주는 간단한 tool 세 개

## 로컬 테스트 Client 생성

AgentCore Runtime에 배포하기 전에 MCP server를 로컬에서 테스트할 client를 생성합니다.

In [ ]:
%%writefile my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

### Testing Locally

MCP server를 로컬에서 테스트하려면 다음을 수행합니다.

1. **Terminal 1**: MCP server 시작
   ```bash
   python mcp_server.py
   ```
   
2. **Terminal 2**: test client 실행
   ```bash
   python my_mcp_client.py
   ```

output에 세 가지 tool 목록이 표시됩니다.

## 인증을 위한 Amazon Cognito 설정

AgentCore Runtime에는 인증이 필요합니다. Amazon Cognito를 사용하여 배포된 MCP server에 액세스할 JWT token을 제공합니다.

In [ ]:
import sys

sys.path.insert(0, "../..")
from utils import setup_cognito_user_pool

print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR repository를 자동으로 생성하도록 starter toolkit을 구성합니다.

configure 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

## AgentCore Runtime에 MCP Server 시작

Dockerfile이 준비되었으므로 MCP server를 AgentCore Runtime에 시작합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## Remote Access용 구성 저장

배포된 MCP server를 호출하기 전에 쉽게 가져올 수 있도록 Agent ARN과 Cognito 구성을 AWS Systems Manager Parameter Store 및 AWS Secrets Manager에 저장합니다.

In [ ]:
import boto3
import json

ssm_client = boto3.client("ssm", region_name=region)
secrets_client = boto3.client("secretsmanager", region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name="mcp_server/cognito/credentials",
        Description="Cognito credentials for MCP server",
        SecretString=json.dumps(cognito_config),
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId="mcp_server/cognito/credentials",
        SecretString=json.dumps(cognito_config),
    )
    print("✓ Cognito credentials updated in Secrets Manager")

agent_arn_response = ssm_client.put_parameter(
    Name="/mcp_server/runtime/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for MCP server",
    Overwrite=True,
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

In [ ]:
import httpx


def stop_runtime_session_oauth(agent_arn, session_id, bearer_token, region):
    """OAuth bearer token으로 runtime 세션을 중지합니다."""
    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/stopruntimesession"

    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
    }

    body = {
        "runtimeSessionId": session_id,
        "agentRuntimeArn": agent_arn,
        "qualifier": "DEFAULT",
    }

    response = httpx.post(url, headers=headers, json=body, timeout=30.0)
    return response


print("✅ Helper function defined for OAuth session stopping")

### Session Lifecycle 시연: Session 중지

Runtime이 배포되었으므로 session 중지를 시연합니다. custom session ID로
MCP server를 호출한 다음 OAuth 인증을 사용하여 중지합니다.

In [ ]:
import uuid
from datetime import timedelta
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# custom session ID 생성
demo1_session_id = str(uuid.uuid4())
print(f"📝 Demo 1 - Generated mcpSessionId: {demo1_session_id}")

# header 준비
encoded_arn = launch_result.agent_arn.replace(":", "%3A").replace("/", "%2F")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

headers = {
    "authorization": f"Bearer {cognito_config['bearer_token']}",
    "Content-Type": "application/json",
    "Mcp-Session-Id": demo1_session_id,
}


# MCP server 호출
async def test_session():
    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read,
        write,
        _,
    ):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"✅ Session active with {len(tools.tools)} tools")


await test_session()

# OAuth를 사용하여 session 중지
print(f"\n🛑 Stopping session '{demo1_session_id}'...")
response = stop_runtime_session_oauth(launch_result.agent_arn, demo1_session_id, cognito_config["bearer_token"], region)
print(f"✅ Session stopped (HTTP {response.status_code})")
print(f"   Response: {response.text}")
print("   MicroVM resources released")
print("💡 Runtime remains alive for new sessions")

# 참고: 위 log에 'Session termination failed: 404'가 표시될 수 있음
# 이는 예상된 동작으로, 이미 session을 중지한 후 MCP client가 자동 정리를 시도하기 때문임
# 중요한 부분은 명시적 stop_runtime_session_oauth 호출의 HTTP 200 응답임

## 원격 테스트 Client 생성

배포된 MCP server를 테스트할 client를 생성합니다. 이 client는 AWS에서 필요한 credentials를 가져와 배포된 server에 연결합니다.

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
import base64
import time
from boto3.session import Session
from datetime import timedelta
import traceback

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """refresh token으로 access token을 갱신합니다."""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """token 만료 시간을 확인하고 필요하면 갱신합니다."""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except Exception as e:
        print("🔄 Invalid token, refreshing...", e)
        traceback.print_exc()
        return get_refresh_token(client_id, refresh_token, region)

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        # 필요한 경우 token 검증 및 갱신
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## 배포된 MCP Server 테스트

remote client를 사용하여 배포된 MCP server를 테스트합니다.

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## MCP Tool 원격 호출

tool 목록을 표시할 뿐 아니라 직접 호출하여 전체 MCP 기능을 보여주는 향상된 client를 생성합니다.

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
import base64
import time
import uuid
import httpx
from boto3.session import Session
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_refresh_token(client_id, refresh_token, region):
    """refresh token으로 access token을 갱신합니다."""
    cognito_client = boto3.client('cognito-idp', region_name=region)
    auth_response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='REFRESH_TOKEN_AUTH',
        AuthParameters={'REFRESH_TOKEN': refresh_token}
    )
    return auth_response['AuthenticationResult']['AccessToken']

def get_valid_token(bearer_token, client_id, refresh_token, region):
    """token 만료 시간을 확인하고 필요하면 갱신합니다."""
    try:
        payload = bearer_token.split('.')[1]
        payload += '=' * (4 - len(payload) % 4)
        decoded = json.loads(base64.b64decode(payload))
        
        current_time = int(time.time())
        if decoded['exp'] - current_time < 300:
            print("🔄 Token expiring soon, refreshing...")
            new_token = get_refresh_token(client_id, refresh_token, region)
            print("✓ Token refreshed successfully")
            return new_token
        
        return bearer_token
    except:
        print("�� Invalid token, refreshing...")
        return get_refresh_token(client_id, refresh_token, region)

def stop_runtime_session_oauth(agent_arn, session_id, bearer_token, region):
    """HTTP POST와 OAuth bearer token으로 runtime 세션을 중지합니다."""
    # URL path에 사용할 ARN encoding
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/stopruntimesession"
    
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    body = {
        "runtimeSessionId": session_id,
        "agentRuntimeArn": agent_arn,
        "qualifier": "DEFAULT"
    }
    
    response = httpx.post(url, headers=headers, json=body, timeout=30.0)
    return response

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        refresh_token = parsed_secret['refresh_token']
        client_id = parsed_secret['client_id']
        print("✓ Retrieved credentials from Secrets Manager")
        
        bearer_token = get_valid_token(bearer_token, client_id, refresh_token, region)
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    
    # custom session ID 생성
    mcp_session_id = str(uuid.uuid4())
    print(f"\n📝 Generated custom mcpSessionId: {mcp_session_id}")
    
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "Mcp-Session-Id": mcp_session_id
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream, write_stream, _
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("✓ MCP session initialized")
                
                tool_result = await session.list_tools()
                print(f"\n📋 Found {len(tool_result.tools)} tools")
                
                # tool 테스트
                print("\n🧪 Testing tools...")
                add_result = await session.call_tool(name="add_numbers", arguments={"a": 5, "b": 3})
                print(f"   add_numbers(5, 3) = {add_result.content[0].text}")
                
                multiply_result = await session.call_tool(name="multiply_numbers", arguments={"a": 4, "b": 7})
                print(f"   multiply_numbers(4, 7) = {multiply_result.content[0].text}")
                
                greet_result = await session.call_tool(name="greet_user", arguments={"name": "Alice"})
                print(f"   greet_user('Alice') = {greet_result.content[0].text}")
                
                print("\n✅ MCP tool testing completed!")
        
        # OAuth bearer token을 사용하여 session 중지
        print(f"\n🛑 Stopping session '{mcp_session_id}' (OAuth)...")
        response = stop_runtime_session_oauth(agent_arn, mcp_session_id, bearer_token, region)
        
        if response.status_code == 200:
            print(f"✅ Session stopped — microVM resources released")
            print(f"💡 Runtime remains alive for new sessions")
        else:
            print(f"⚠️  Status {response.status_code}: {response.text}")
                
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    asyncio.run(main())

## Tool 호출 테스트

MCP tool을 직접 호출하여 테스트합니다.

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

### Session Lifecycle 시연: 테스트 사이에 Session 중지

서로 다른 테스트 방식 사이에서 session을 중지하는 방법을 시연합니다.

In [ ]:
# pattern을 보여주기 위해 다른 session을 생성하고 중지
demo2_session_id = str(uuid.uuid4())
print(f"📝 Demo 2 - Generated mcpSessionId: {demo2_session_id}")

headers2 = {
    "authorization": f"Bearer {cognito_config['bearer_token']}",
    "Content-Type": "application/json",
    "Mcp-Session-Id": demo2_session_id,
}


async def test_session2():
    async with streamablehttp_client(mcp_url, headers2, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read,
        write,
        _,
    ):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(f"✅ Session {demo2_session_id} created")


await test_session2()

print(f"🛑 Stopping session '{demo2_session_id}'...")
response = stop_runtime_session_oauth(launch_result.agent_arn, demo2_session_id, cognito_config["bearer_token"], region)
print(f"✅ Session stopped (HTTP {response.status_code})")

## 다음 단계

MCP server를 AgentCore Runtime에 성공적으로 배포했으므로 다음 작업을 수행할 수 있습니다.

1. **Tool 추가**: MCP server에 tool 추가
2. **Custom 인증**: custom JWT authorizer 구현
3. **통합**: 다른 AgentCore service와 통합

## Session Lifecycle 모범 사례

AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 책정됩니다. 원치 않는 비용을 방지하려면 session을 명시적으로 중지하거나 적절한 idle timeout을 구성하여 session이 종료되도록 하는 것이 좋습니다.

비용을 효과적으로 관리하려면 다음을 수행합니다.

- **idle timeout 구성**: session을 생성할 때 적절한 idle timeout을 설정하여 비활성 session을 자동으로 중지합니다. 사용 사례에 맞는 값(예: 개발/테스트에는 짧게, production workload에는 길게)을 선택합니다.
- **완료 후 session 중지**: Runtime은 새 session에 사용할 수 있도록 유지하면서 `stop_runtime_session`으로 특정 session의 microVM 리소스를 해제합니다.

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 원치 않는 비용을 방지하도록 Runtime을 먼저 삭제한 다음 ECR repository와 Secrets Manager secret 같은 지원 리소스를 정리합니다.

In [ ]:
# --- 리소스 정리 ---
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# 1단계: credential 노출 시간을 최소화하도록 Secrets Manager secret을 먼저 삭제
try:
    secrets_client.delete_secret(SecretId="mcp_server/cognito/credentials", ForceDeleteWithoutRecovery=True)
    print("✅ Secrets Manager secret deleted")
except secrets_client.exceptions.ResourceNotFoundException:
    print("ℹ️  Secrets Manager secret not found")

# 2단계: Parameter Store parameter 삭제
try:
    ssm_client.delete_parameter(Name="/mcp_server/runtime/agent_arn")
    print("✅ Parameter Store parameter deleted")
except ssm_client.exceptions.ParameterNotFound:
    print("ℹ️  Parameter Store parameter not found")

# 3단계: 비용 발생을 중지하도록 agent runtime 삭제
# AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 책정됨
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Agent runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete agent runtime: {e}")

# 4단계: ECR repository 삭제
try:
    ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

print("\n✅ Cleanup completed successfully!")

# 🎉 축하합니다!

다음 작업을 성공적으로 완료했습니다.

✅ custom tool로 **MCP server 생성**  
✅ MCP client로 **로컬 테스트**  
✅ Amazon Cognito로 **인증 설정**  
✅ AgentCore Runtime을 사용하여 **AWS에 배포**  
✅ 적절한 인증으로 **원격 호출**  
✅ **MCP 개념과 best practice 학습**  

이제 MCP server가 Amazon Bedrock AgentCore Runtime에서 실행되고 있으며 production에서 사용할 준비가 되었습니다.

## 요약

이 튜토리얼에서는 다음 방법을 학습했습니다.
- FastMCP를 사용하여 MCP server 구축
- AgentCore 호환성을 위한 stateless HTTP transport 구성
- Amazon Cognito로 JWT 인증 설정
- AWS에 MCP server를 배포하고 관리
- 로컬 및 원격 테스트
- tool 호출에 MCP client 사용

배포된 MCP server를 이제 더 큰 AI 애플리케이션과 workflow에 통합할 수 있습니다.